In [2]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd

VIDEO_IDS = [
    "nPUz4DXtYzs", "UyuWysU4v9g", "sPTNYp16nxc", "rSO7gZRDrOM",
    "xSILWfgpfpw", "CGf1qBHETzw", "Q8hUYTE6QUw", "KL9Y0A1pmjE",
    "eVtVg5Nl7ag", "j-oRcnx_beM",
    "AKmBujNy14o", "1CmzeqE3bFw", "2g2QQmCtzjU", "Q3U06YIV_Q0",
    "J8-1vA505jQ", "z_08cYldDco", "1-oom8s1mFg", "JEqaTyGu5u4",
    "HiAMmO4e6pE", "2aiHwEDEUTU"
]

ytt_api = YouTubeTranscriptApi()
all_transcripts = []
for vid in VIDEO_IDS:
    try:
        transcript = ytt_api.fetch(vid, languages=['th', 'en'])
        text = ' '.join([snippet.text for snippet in transcript])
        all_transcripts.append(text)
        print(f"✓ Downloaded: {vid}")
    except Exception as e:
        print(f"✗ Failed: {vid} — {e}")

df = pd.DataFrame({'message': all_transcripts})
print(f"\nTotal transcripts loaded: {len(df)}")
df.head()

✓ Downloaded: nPUz4DXtYzs
✓ Downloaded: UyuWysU4v9g
✓ Downloaded: sPTNYp16nxc
✓ Downloaded: rSO7gZRDrOM
✓ Downloaded: xSILWfgpfpw
✓ Downloaded: CGf1qBHETzw
✓ Downloaded: Q8hUYTE6QUw
✓ Downloaded: KL9Y0A1pmjE
✓ Downloaded: eVtVg5Nl7ag
✓ Downloaded: j-oRcnx_beM
✓ Downloaded: AKmBujNy14o
✓ Downloaded: 1CmzeqE3bFw
✓ Downloaded: 2g2QQmCtzjU
✓ Downloaded: Q3U06YIV_Q0
✓ Downloaded: J8-1vA505jQ
✓ Downloaded: z_08cYldDco
✓ Downloaded: 1-oom8s1mFg
✓ Downloaded: JEqaTyGu5u4
✓ Downloaded: HiAMmO4e6pE
✓ Downloaded: 2aiHwEDEUTU

Total transcripts loaded: 20


,message
0,ทุกคนคือกูมีความคิดจะเปิดข้าวมันไก่เอ้ย เอาเปิ...
1,เรื่องราวมันอยากจะเล่าให้ทุกคนฟังมากเลย ทุกคนแ...
2,สำหรับเนื้อหาหลักของเราในวันนี้นะครับ เราจะมาพ...
3,สำหรับเนื้อหาหลักของเราในวันนี้นะครับ เราจะมาพ...
4,สำหรับเนื้อหาหลักของเราในวันนี้นะครับก็ จะมาเล...


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [3]:
df = pd.read_csv('transcript.csv')
texts = df['message'].astype(str).tolist()

print(f"จำนวนประโยคทั้งหมด: {len(texts)}")
print("ตัวอย่างข้อมูล:", texts[0])

จำนวนประโยคทั้งหมด: 5130
ตัวอย่างข้อมูล: ทุกคน คือ กู มี ความคิด จะ เปิด ข้าวมันไก่ เอ้ย


In [4]:
tokenizer = Tokenizer(filters='')
tokenizer.fit_on_texts(texts)

total_words = len(tokenizer.word_index) + 1
print(f"จำนวนคำศัพท์ทั้งหมด (Vocab size): {total_words}")

input_sequences = []
for line in texts:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"จำนวน sequences ที่สร้างได้: {len(input_sequences)}")

จำนวนคำศัพท์ทั้งหมด (Vocab size): 7879
จำนวน sequences ที่สร้างได้: 140356


In [7]:
max_sequence_len = max([len(x) for x in input_sequences])
print(f"Max Sequence: {max_sequence_len}")

input_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

input_sequences = np.array(input_sequences)
X, labels = input_sequences[:,:-1], input_sequences[:,-1]

y = to_categorical(labels, num_classes=total_words)

print(f"Shape X: {X.shape}")
print(f"Shape y: {y.shape}")

Max Sequence: 552
Shape X: (140356, 551)
Shape y: (140356, 7879)


In [11]:
model = Sequential()

model.add(Embedding(input_dim=total_words, output_dim=100, input_shape=(max_sequence_len-1,)))

model.add(Bidirectional(LSTM(150, return_sequences=False)))
model.add(Dropout(0.2))

model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

c:\Users\nawatpim\anaconda3\envs\my_python\lib\site-packages\keras\src\layers\core\embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 551, 100)       │       787,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 300)            │       301,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7879)           │     2,371,579 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,460,679 (13.20 MB)

 Trainable params: 3,460,679 (13.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X, y, epochs=150, batch_size=64, verbose=1)

Epoch 1/50
  30/2194 ━━━━━━━━━━━━━━━━━━━━ 58:22 2s/step - accuracy: 0.0228 - loss: 8.5355

KeyboardInterrupt: 